# NB_04: Kiến Trúc Phân Cấp Đồng Nhất `CW-SSN` & Hàm Mất Mát Vận Chuyển Tối Ưu Phổ `SpectralWassersteinLoss`
**Dự án mục tiêu:** *Complex Wasserstein State-Space Network (CW-SSN)* — Nộp tạp chí Q1 **IEEE TPAMI**  
**Giai đoạn:** Giai đoạn 2 (Tuần 5) | **Phân bổ phần cứng:** Tài khoản Kaggle T4x2 #2 | **Thời gian phiên:** 5 – 6 giờ

---

## 1. Mục Tiêu Chiến Lược & Gatekeeping KPIs
1. **Xếp chồng đồng nhất khối `CWSSBlock`** tạo thành mạng phân cấp 4 tầng `CW-SSN-T` (`5.81M` tham số) và `CW-SSN-B` (`7.95M` tham số).
2. **Thiết lập hàm mục tiêu hợp nhất `SpectralWassersteinLoss`**:
   $$\mathcal{L}_{\text{total}}(\hat{I}, I) = \|\hat{I} - I\|_1 + \lambda_{W_2} \mathcal{W}_2^2\big(\mathcal{P}(\hat{I}), \mathcal{P}(I)\big) + \lambda_{\text{freq}} \|\mathcal{F}(\hat{I}) - \mathcal{F}(I)\|_1$$
3. **Gatekeeping KPI:** Chạy Pilot Run đối chứng giữa hàm mất mát $L_1$ thuần và `SpectralWassersteinLoss`: chứng minh hàm mục tiêu vận chuyển tối ưu phổ hội tụ mượt mà và mang lại mức tăng **$\ge +0.45\text{ dB PSNR}$** trên cấu trúc chi tiết tần số cao.

In [ ]:
import os
import sys
import time
import math
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure repository root is on sys.path (works on Kaggle, Colab Pro, and Local)
REPO_URL = "https://github.com/hannhu11/Complex-Wasserstein-State-Space-Network-CW-SSN.git"
REPO_DIR = Path("Complex-Wasserstein-State-Space-Network-CW-SSN")

if not Path("models/cw_ssn.py").exists():
    if REPO_DIR.exists() and (REPO_DIR / "models/cw_ssn.py").exists():
        sys.path.insert(0, str(REPO_DIR.resolve()))
    elif Path("..").resolve().joinpath("models/cw_ssn.py").exists():
        sys.path.insert(0, str(Path("..").resolve()))
    else:
        print(f"[Setup] Cloning {REPO_URL} ...")
        os.system(f"git clone --depth 1 {REPO_URL}")
        if REPO_DIR.exists():
            sys.path.insert(0, str(REPO_DIR.resolve()))
else:
    sys.path.insert(0, str(Path(".").resolve()))

from models.cw_ssn import (
    ComplexWassersteinSSM2D,
    CWSSBlock,
    CWSSN,
    CWSSOAdapter,
    SpectralWassersteinLoss,
    build_cw_ssn_tiny,
    build_cw_ssn_base,
    rotational_2x2_Scan_parallel,
    rotational_2x2_scan_sequential,
)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Environment] PyTorch: {torch.__version__} | Device: {DEVICE} | GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(idx)
        print(f"  -> GPU {idx}: {p.name} ({p.total_memory / (1024**3):.2f} GB VRAM)")

## 2. Thí Nghiệm Pilot Run: So Sánh Hội Tụ Giữa Hàm Mất Mát $L_1$ Thuần Và `SpectralWassersteinLoss`

In [ ]:
def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(torch.clamp(pred, 0.0, 1.0), torch.clamp(target, 0.0, 1.0)).item()
    if mse < 1e-10:
        return 50.0
    return 10.0 * math.log10(1.0 / mse)

# Create structured multi-frequency pilot restoration pair
seed_everything(2026)
B, C, H, W = 2, 3, 32, 32
grid_y, grid_x = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing="ij")
clean_batch = (
    0.5
    + 0.25 * torch.sin(2 * math.pi * 4 * grid_x) * torch.cos(2 * math.pi * 4 * grid_y)
    + 0.15 * torch.sin(2 * math.pi * 8 * grid_x)
).unsqueeze(0).unsqueeze(0).repeat(B, C, 1, 1).to(DEVICE)
clean_batch = torch.clamp(clean_batch, 0.0, 1.0)
degraded_batch = torch.clamp(clean_batch + torch.randn_like(clean_batch) * 0.12, 0.0, 1.0)

def run_pilot_training(use_spectral_w2_loss: bool, steps: int = 18) -> float:
    seed_everything(42)
    net = CWSSN(base_dim=24, depths=(1, 1, 2, 1), refinement_blocks=1).to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=0.005, momentum=0.9, weight_decay=1e-4)
    crit = SpectralWassersteinLoss(lambda_w2=0.45, lambda_freq=0.25) if use_spectral_w2_loss else nn.L1Loss()

    for _ in range(steps):
        opt.zero_grad()
        pred = net(degraded_batch)
        loss = crit(pred, clean_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
        opt.step()

    with torch.no_grad():
        final_pred = net(degraded_batch)
    psnr_val = compute_psnr(final_pred, clean_batch)
    del net, opt
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return psnr_val

psnr_l1_only = run_pilot_training(use_spectral_w2_loss=False, steps=18)
psnr_full_loss = run_pilot_training(use_spectral_w2_loss=True, steps=18)
delta_psnr = psnr_full_loss - psnr_l1_only
print(f"[Pilot Result] Pure L1 Loss PSNR:                 {psnr_l1_only:.2f} dB")
print(f"[Pilot Result] SpectralWassersteinLoss PSNR:      {psnr_full_loss:.2f} dB (Delta: {delta_psnr:+.2f} dB)")
assert delta_psnr >= 0.45, f"Expected >= +0.45 dB gain from SpectralWassersteinLoss, got {delta_psnr:+.2f} dB"
print("[GATEKEEPER PASSED] CW-SSN homogeneous hierarchy and SpectralWassersteinLoss verified.")